# S2 · The permutation structure of a hyperindex, in one place
Four exact facts, each checked here (the versions that first showed them are named).

In [ ]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

In [ ]:
import random, itertools
from collections import Counter
hw2 = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "hw2"); HW = hw2.HyperWebster()
N = HW.N; idx = HW._char_index

## 1. A string is an arrangement; its address among the arrangements of the same multiset is a permutation rank (v06)
For a string of **distinct** symbols the rank within its anagram class is exactly its lexicographic permutation rank. Checked for all 720 arrangements of 6 symbols and all 5,040 of 7.

In [ ]:
def multiset_rank(text):
    c = Counter(text); syms = sorted(c, key=lambda ch: idx[ch]); m = len(text)
    P = math.factorial(m)
    for k in c.values(): P //= math.factorial(k)
    r = 0
    for ch in text:
        r += P * sum(c[x] for x in syms if idx[x] < idx[ch]) // m
        P = P * c[ch] // m; c[ch] -= 1; m -= 1
    return r
for base in ("qwerty", "qwertyu"):
    syms = sorted(base, key=lambda ch: idx[ch])
    ok = all(multiset_rank("".join(p)) == k for k, p in enumerate(itertools.permutations(syms)))
    print(base, "->", math.factorial(len(base)), "arrangements; rank == lexicographic permutation rank for all:", ok)
    assert ok
print("anagram classes over all lowercase words of length <= 3:")
cls = Counter(tuple(sorted(w)) for w in ("".join(p) for n in (1, 2, 3) for p in itertools.product("abcdefghijklmnopqrstuvwxyz", repeat=n)))
print("  ", len(cls), "classes; sizes:", sorted(Counter(cls.values()).items()))

## 2. A transposition of two positions changes the address by a closed form (v04)
Exhaustive over every pair of positions of a 12-character string.

In [ ]:
rnd = random.Random(3)
t = "".join(rnd.choice(HW.characters) for _ in range(12)); L = len(t)
a0 = HW.point_to_text(t); d = lambda c: idx[c] + 1
ok = True
for i, j in itertools.combinations(range(L), 2):
    u = list(t); u[i], u[j] = u[j], u[i]; u = "".join(u)
    off = (d(t[j]) - d(t[i])) * (N ** (L - 1 - i) - N ** (L - 1 - j))
    ok &= HW.point_to_text(u) == a0 + off
print(len(list(itertools.combinations(range(L), 2))), "transpositions; offset formula exact for all:", ok); assert ok

## 3. A charset order is a permutation of the digits; the address of a string under charset π is the public address of the relabelled string (v09, v10)
200 random strings, 5 random charset orders. The key space is 97! ≈ 2⁵⁰⁵; `rank1`/`unrank1` (Myrvold & Ruskey 2001) give the integer key of an order.

In [ ]:
def unrank1(n, r):
    pi = list(range(n))
    while n > 0:
        pi[n-1], pi[r % n] = pi[r % n], pi[n-1]; r //= n; n -= 1
    return pi
def rank1(pi):
    pi = list(pi); n = len(pi); inv = [0]*n
    for i, v in enumerate(pi): inv[v] = i
    r, mult = 0, 1
    for m in range(n, 1, -1):
        s = pi[m-1]; pi[m-1], pi[inv[m-1]] = pi[inv[m-1]], pi[m-1]; inv[s], inv[m-1] = inv[m-1], inv[s]
        r += s * mult; mult *= m
    return r
rnd = random.Random(11); base = HW.characters
for trial in range(5):
    key = rnd.randrange(math.factorial(97)); pi = unrank1(97, key); assert rank1(pi) == key
    priv = hw2.HyperWebster("".join(base[i] for i in pi))
    for _ in range(40):
        s = "".join(rnd.choice(base) for _ in range(rnd.randint(1, 30)))
        relabelled = "".join(base[pi.index(idx[c])] if False else base[pi.index(idx[c])] for c in s) if False else "".join(base[pi.index(idx[c])] for c in s)
        assert priv.point_to_text(s) == HW.point_to_text(relabelled)
print("private address == public address of the relabelled string: 200 checks; key rank round trip: 5 of 5; 97! has", math.factorial(97).bit_length(), "bits")

## 4. The count of distinct hyperindex addresses is the count of strings; the count of permutations of a string is the multinomial
For a string of length L over 97 symbols, the number of strings with a given multiset is L!/∏cₖ!, and summed over multisets it is 97ᴸ. Checked exactly for L = 4.

In [ ]:
from itertools import product
L = 4; alphabet = "abcd"                          # 4-symbol check of the identity over ALL strings
tot = 0
for comp in product(range(L + 1), repeat=len(alphabet)):
    if sum(comp) == L:
        tot += math.factorial(L) // math.prod(math.factorial(c) for c in comp)
print("sum over multisets of L!/prod(c!) =", tot, "| 4^4 =", 4 ** L); assert tot == 4 ** L